**##** **IMPLEMENTING SELF ATTENTION WITH TRAINABLE WEIGHTS**

In [1]:
# Import the PyTorch library for tensor operations
import torch

# Create a tensor containing embedding vectors for six input words
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],  # "Your" word embedding (x¹)
   [0.55, 0.87, 0.66],  # "journey" word embedding (x²)
   [0.57, 0.85, 0.64],  # "starts" word embedding (x³)
   [0.22, 0.58, 0.33],  # "with" word embedding (x⁴)
   [0.77, 0.25, 0.10],  # "one" word embedding (x⁵)
   [0.05, 0.80, 0.55]]  # "step" word embedding (x⁶)
)

Note that in GPT-like models, the input and output dimensions are usually the same.

But for illustration purposes, to better follow the computation, we choose different input **(d_in=3)**
and output **(d_out=2)** dimensions here.



In [2]:
# Select the second input token embedding ("journey")
x_2 = inputs[1]  # A

# Get the input embedding dimension (number of features)
d_in = inputs.shape[1]  # B

# Set the output dimension for Query, Key, and Value vectors
d_out = 2  # C

In [3]:
# Set a random seed to generate reproducible weight values
torch.manual_seed(123)

# Initialize the Query weight matrix (WQ)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# Initialize the Key weight matrix (WK)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# Initialize the Value weight matrix (WV)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [4]:
print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [5]:
print(W_key)

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])


In [6]:
print(W_value)

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


In [7]:
# Compute the Query vector for the second input token
query_2 = x_2 @ W_query

# Compute the Key vector for the second input token
key_2 = x_2 @ W_key

# Compute the Value vector for the second input token
value_2 = x_2 @ W_value

# Display the generated Query vector
print(query_2)

tensor([0.4306, 1.4551])


In [8]:
# Generate Key vectors for all input tokens
keys = inputs @ W_key

# Generate Value vectors for all input tokens
values = inputs @ W_value

# Generate Query vectors for all input tokens
queries = inputs @ W_query

# Display the shape of the Key matrix
print("keys.shape:", keys.shape)

# Display the shape of the Value matrix
print("values.shape:", values.shape)

# Display the shape of the Query matrix
print("queries.shape:", queries.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
queries.shape: torch.Size([6, 2])


In [9]:
# Select the Key vector corresponding to the second input token
keys_2 = keys[1]  # A

# Compute the attention score between the second Query and second Key
attn_score_22 = query_2.dot(keys_2)

# Display the calculated attention score
print(attn_score_22)

tensor(1.8524)


In [10]:
# Compute attention scores between the second Query and all Key vectors
attn_scores_2 = query_2 @ keys.T  # All attention scores for the given query

# Display the attention scores
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [11]:
# Compute pairwise attention scores between all Query and Key vectors
attn_scores = queries @ keys.T  # Attention score matrix (ω)

# Display the attention score matrix
print(attn_scores)

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])


In [12]:
# Get the dimension of the Key vectors
d_k = keys.shape[-1]

# Apply scaled softmax to convert attention scores into attention weights
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)

# Display the attention weights for the second query
print(attn_weights_2)

# Display the Key vector dimension used for scaling
print(d_k)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
2


In [13]:
# Compute the context vector as the weighted sum of all Value vectors
context_vec_2 = attn_weights_2 @ values

# Display the context vector for the second query token
print(context_vec_2)

tensor([0.3061, 0.8210])


**In Structured Manner With Python Class**

In [14]:
# Import the PyTorch neural network module
import torch.nn as nn

# Define a Self-Attention layer from scratch
class SelfAttention_v1(nn.Module):

    # Initialize the self-attention layer and its learnable parameters
    def __init__(self, d_in, d_out):
        super().__init__()

        # Create the Query weight matrix
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))

        # Create the Key weight matrix
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))

        # Create the Value weight matrix
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    # Define the forward pass of the self-attention mechanism
    def forward(self, x):

        # Generate Key vectors from the input embeddings
        keys = x @ self.W_key

        # Generate Query vectors from the input embeddings
        queries = x @ self.W_query

        # Generate Value vectors from the input embeddings
        values = x @ self.W_value

        # Compute attention scores between all Query and Key pairs
        attn_scores = queries @ keys.T  # ω

        # Convert attention scores into attention weights using scaled softmax
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        # Compute context vectors as weighted sums of Value vectors
        context_vec = attn_weights @ values

        # Return the final context vectors
        return context_vec

In [15]:
# Set a random seed to ensure reproducible weight initialization
torch.manual_seed(123)

# Create an instance of the SelfAttention_v1 layer
sa_v1 = SelfAttention_v1(d_in, d_out)

# Pass the input embeddings through the self-attention layer
# to compute context vectors for all input tokens
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [16]:
# Define a self-attention layer using PyTorch Linear layers
class SelfAttention_v2(nn.Module):

    # Initialize the layer and optionally include bias terms
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        # Use nn.Linear to automatically perform matrix multiplication,
        # manage learnable weights (and optional biases), and simplify the code
        # Linear layer to generate Query vectors
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Key vectors
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Value vectors
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    # Define the forward pass of the self-attention mechanism
    def forward(self, x):

        # Generate Key vectors from the input embeddings
        keys = self.W_key(x)

        # Generate Query vectors from the input embeddings
        queries = self.W_query(x)

        # Generate Value vectors from the input embeddings
        values = self.W_value(x)

        # Compute attention scores between all Query and Key pairs
        attn_scores = queries @ keys.T

        # Convert attention scores into attention weights using scaled softmax
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        # Compute context vectors as weighted sums of Value vectors
        context_vec = attn_weights @ values

        # Return the final context vectors
        return context_vec

In [17]:
# Set a random seed to ensure reproducible weight initialization
torch.manual_seed(789)

# Create an instance of the SelfAttention_v2 layer
sa_v2 = SelfAttention_v2(d_in, d_out)

# Pass the input embeddings through the self-attention layer to generate context vectors for all input tokens
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


**##** **IMPLEMENTING CASUAL ATTENTION**

In [18]:
# Import the PyTorch library for tensor operations
import torch

# Create a tensor containing embedding vectors for six input words
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],  # "Your" word embedding (x¹)
   [0.55, 0.87, 0.66],  # "journey" word embedding (x²)
   [0.57, 0.85, 0.64],  # "starts" word embedding (x³)
   [0.22, 0.58, 0.33],  # "with" word embedding (x⁴)
   [0.77, 0.25, 0.10],  # "one" word embedding (x⁵)
   [0.05, 0.80, 0.55]]  # "step" word embedding (x⁶)
)

In [19]:
# Generate Query vectors for all input tokens
queries = sa_v2.W_query(inputs)  # A

# Generate Key vectors for all input tokens
keys = sa_v2.W_key(inputs)

# Compute attention scores between all Query and Key pairs
attn_scores = queries @ keys.T

# Apply scaled softmax to convert attention scores into attention weights
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5, dim=1
)

# Display the attention weight matrix
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [20]:
# Get the number of tokens in the sequence (sequence length)
context_length = attn_scores.shape[0]

# Create a lower-triangular mask for causal (autoregressive) attention
# torch.ones(...) creates a matrix filled with 1s
# torch.tril(...) keeps only the lower triangle (including the diagonal)
# 1 = token is allowed to attend
# 0 = token is not allowed to attend
# This prevents a token from looking at future tokens during attention computation
mask_simple = torch.tril(torch.ones(context_length, context_length))

# Display the causal attention mask
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [21]:
# Apply the causal mask to the attention weights
# Multiplying by the mask keeps allowed attention weights unchanged (1 × weight)
# and sets attention to future tokens to zero (0 × weight = 0)
masked_simple = attn_weights * mask_simple

# Display the masked attention weights
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [22]:
# Compute the sum of attention weights in each row
# keepdim=True preserves the matrix shape for broadcasting during division
row_sums = masked_simple.sum(dim=1, keepdim=True)

# Renormalize the masked attention weights
# This ensures that each row sums to 1 after masking
masked_simple_norm = masked_simple / row_sums

# Display the normalized attention weights
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [23]:
# Create an upper-triangular mask to identify future token positions
# diagonal=1 keeps the main diagonal unchanged and masks only future tokens
# 1 = position to be masked, 0 = position to keep
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

# Replace attention scores at masked positions with -infinity
# During softmax, exp(-∞) becomes 0, preventing attention to future tokens
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)

# Display the masked attention score matrix
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [24]:
# Apply scaled softmax to the masked attention scores
# Dividing by √d_k keeps the score variance stable
# Softmax converts the scores into attention probabilities
# Positions with -∞ become 0 after softmax, preventing attention to future tokens
attn_weights = torch.softmax(
    masked / keys.shape[-1]**0.5,
    dim=1
)

# Display the final masked attention weights
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [25]:
# Set a random seed for reproducible dropout results
torch.manual_seed(123)

# Create a Dropout layer with a dropout rate of 50%
# During training, randomly sets 50% of elements to zero
# The remaining elements are scaled by 1/(1-p) to maintain the expected value
dropout = torch.nn.Dropout(0.5)  # A

# Create a 6×6 tensor filled with ones
# Used to demonstrate the effect of dropout
example = torch.ones(6, 6)  # B

# Apply dropout and display the result
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [26]:
# Set a random seed to ensure reproducible dropout behavior
torch.manual_seed(123)

# Apply dropout to the attention weights
# Randomly removes 50% of the attention connections during training
# This prevents the model from relying too heavily on specific tokens
# The remaining attention weights are scaled to preserve their expected value
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


**### IMPLEMENTING A COMPACT CAUSAL ATTENTION CLASS**

In [27]:
# Create a batch by stacking two copies of the input sequence
# dim=0 adds a new batch dimension at the beginning
batch = torch.stack((inputs, inputs), dim=0)

# Display the shape of the batched input tensor
print(batch.shape)

torch.Size([2, 6, 3])


In [28]:
# Define a causal self-attention layer for autoregressive models
class CausalAttention(nn.Module):

    # Initialize the attention layer and its learnable parameters
    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()

        # Store the output dimension of Query, Key, and Value vectors
        self.d_out = d_out

        # Linear layer to generate Query vectors
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Key vectors
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Value vectors
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Dropout layer to reduce overfitting during training
        self.dropout = nn.Dropout(dropout)  # New

        # Create and store a causal mask
        # Upper-triangular positions (future tokens) are marked with 1
        # register_buffer stores the mask with the model but does not train it
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )  # New

    # Define the forward pass
    def forward(self, x):

        # Extract batch size, sequence length, and input dimension
        b, num_tokens, d_in = x.shape  # New batch dimension b

        # Generate Key vectors
        keys = self.W_key(x)

        # Generate Query vectors
        queries = self.W_query(x)

        # Generate Value vectors
        values = self.W_value(x)

        # Compute attention scores for every token pair in each sequence
        # transpose(1, 2) swaps token and feature dimensions of keys
        attn_scores = queries @ keys.transpose(1, 2)

        # Apply the causal mask
        # Future-token positions are replaced with -∞
        # so they receive zero attention after softmax
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf
        )

        # Apply scaled softmax to obtain attention weights
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        # Apply dropout to the attention weights
        attn_weights = self.dropout(attn_weights)

        # Compute context vectors as weighted sums of Value vectors
        context_vec = attn_weights @ values

        # Return the final context vectors
        return context_vec

In [29]:
# Set a random seed to ensure reproducible weight initialization
torch.manual_seed(123)

# Get the sequence length (number of tokens) from the batch
# Used to create the causal attention mask
context_length = batch.shape[1]

# Create a CausalAttention layer
# dropout=0.0 means no attention weights will be dropped
ca = CausalAttention(d_in, d_out, context_length, 0.0)

# Pass the batch through the causal attention layer
# Generates context vectors for every token in every sequence
context_vecs = ca(batch)

# Display the shape of the resulting context vectors
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


**## EXTENDING SINGLE HEAD ATTENTION TO MULTI-HEAD ATTENTION**

In [30]:
# Define a wrapper that combines multiple attention heads
# Multi-head attention allows the model to learn different types of relationships
# between tokens simultaneously (e.g., grammar, meaning, position)
class MultiHeadAttentionWrapper(nn.Module):

    # Initialize multiple independent causal attention heads
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # Create a list of attention heads
        # Each head has its own Query, Key, and Value weight matrices
        # ModuleList ensures all heads are properly registered and trained
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    # Define the forward pass
    def forward(self, x):

        # Pass the input through every attention head independently
        # Each head learns a different attention pattern
        # torch.cat(..., dim=-1) concatenates all head outputs
        # along the feature dimension to create a richer representation
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [31]:
# Set a random seed to ensure reproducible weight initialization
torch.manual_seed(123)

# Get the sequence length (number of tokens) from the batch
# This value is used to create the causal attention mask
context_length = batch.shape[1]  # This is the number of tokens

# Define the input embedding dimension and output dimension per head
d_in, d_out = 3, 2

# Create a Multi-Head Attention module with 2 attention heads
# Each head independently learns different token relationships
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

# Pass the batch through all attention heads
# The outputs from both heads are concatenated along the last dimension
context_vecs = mha(batch)

# Display the generated context vectors
print(context_vecs)

# Display the shape of the final output tensor
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


**### IMPLEMENTING MULTI-HEAD ATTENTION WITH WEIGHT SPLITS**

In [32]:
# Define an efficient Multi-Head Self-Attention layer
# Instead of creating separate attention modules for each head,
# all heads are computed in parallel using tensor reshaping
class MultiHeadAttention(nn.Module):

    # Initialize the multi-head attention layer
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # Ensure the output dimension can be evenly split across all heads
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        # Total output dimension after combining all heads
        self.d_out = d_out

        # Number of attention heads
        self.num_heads = num_heads

        # Dimension handled by each head
        # Example: d_out=8, num_heads=4 => head_dim=2
        self.head_dim = d_out // num_heads

        # Linear layer to generate Query vectors for all heads at once
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Key vectors for all heads at once
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Value vectors for all heads at once
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Final projection layer used to mix information from all heads
        self.out_proj = nn.Linear(d_out, d_out)

        # Dropout applied to attention weights
        self.dropout = nn.Dropout(dropout)

        # Create and store a causal mask
        # Future-token positions are marked with 1
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    # Define the forward pass
    def forward(self, x):

        # Extract batch size, sequence length, and input dimension
        b, num_tokens, d_in = x.shape

        # Generate Query vectors
        keys = self.W_key(x)      # Shape: (b, num_tokens, d_out)

        # Generate Key vectors
        queries = self.W_query(x)

        # Generate Value vectors
        values = self.W_value(x)

        # Split d_out into multiple attention heads
        # Example:
        # Before: (2, 6, 8)
        # After : (2, 6, 4, 2)
        #
        # Why split?
        # Each head learns different relationships between tokens.
        # Head 1 may focus on nearby words,
        # Head 2 may focus on long-range dependencies,
        # Head 3 may focus on grammar,
        # Head 4 may focus on meaning.
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)

        # Split Value vectors into multiple heads
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # Split Query vectors into multiple heads
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Move the head dimension before token dimension
        # Before: (b, num_tokens, num_heads, head_dim)
        # After : (b, num_heads, num_tokens, head_dim)
        #
        # This allows each head to perform attention independently
        keys = keys.transpose(1, 2)

        # Rearrange Query tensor similarly
        queries = queries.transpose(1, 2)

        # Rearrange Value tensor similarly
        values = values.transpose(1, 2)

        # Compute attention scores separately for each head
        #
        # Shape:
        # queries : (b, num_heads, num_tokens, head_dim)
        # keys.T  : (b, num_heads, head_dim, num_tokens)
        #
        # Result:
        # (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # Create a boolean causal mask
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Prevent attention to future tokens
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Scale scores and convert them into attention probabilities
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        # Apply dropout to attention weights
        attn_weights = self.dropout(attn_weights)

        # Compute context vectors for each head
        #
        # Shape:
        # (b, num_heads, num_tokens, head_dim)
        context_vec = (attn_weights @ values)

        # Move heads back after attention computation
        #
        # Before:
        # (b, num_heads, num_tokens, head_dim)
        #
        # After:
        # (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # Merge all heads back together
        #
        # Example:
        # Before: (2, 6, 4, 2)
        # After : (2, 6, 8)
        #
        # This combines information learned by all heads
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )

        # Final linear projection to mix information across heads
        context_vec = self.out_proj(context_vec)

        # Return final multi-head attention output
        return context_vec

In [33]:
# Create a 4-dimensional tensor for demonstrating batched multi-head operations
# Shape: (1, 2, 3, 4)
# 1 = batch size
# 2 = number of attention heads
# 3 = number of tokens
# 4 = features (head dimension) per token
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],  # A
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

In [34]:
# Transpose the last two dimensions of the tensor
# Shape changes from (1, 2, 3, 4) to (1, 2, 4, 3)
# This allows matrix multiplication between token vectors
# within each attention head
print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [35]:
# Extract the first attention head from the first batch
# Shape: (3, 4) -> 3 tokens, each with 4 features
first_head = a[0, 0, :, :]

# Compute pairwise dot-product similarities between all tokens
# Resulting shape: (3, 3)
# Each value represents how similar one token is to another
first_res = first_head @ first_head.T

# Display the attention score matrix for the first head
print("First head:\n", first_res)

# Extract the second attention head from the first batch
# Shape: (3, 4) -> 3 tokens, each with 4 features
second_head = a[0, 1, :, :]

# Compute pairwise dot-product similarities between all tokens
# for the second attention head
second_res = second_head @ second_head.T

# Display the attention score matrix for the second head
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [36]:
# Set a random seed to ensure reproducible weight initialization
torch.manual_seed(123)

# Extract batch size, sequence length, and input dimension from the input tensor
batch_size, context_length, d_in = batch.shape

# Set the total output dimension of the multi-head attention layer
d_out = 2

# Create a Multi-Head Attention layer with 2 attention heads
# Since d_out=2 and num_heads=2, each head gets head_dim=1 feature
mha = MultiHeadAttention(
    d_in,
    d_out,
    context_length,
    0.0,          # No dropout
    num_heads=2
)

# Pass the input batch through the multi-head attention layer
# Each head computes attention independently and their outputs are combined
context_vecs = mha(batch)

# Display the generated context vectors
print(context_vecs)

# Display the shape of the final output tensor
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
